# Module 7: RAG chatbot with citations

**Project:** build a chatbot that answers questions **from your own documents** and **shows its sources**. It must say "I don't know" when the documents do not contain the answer, instead of making something up.

**RAG** stands for *retrieval-augmented generation*: first **retrieve** the most relevant passages (your Module 6 search engine), then ask a language model to **generate** an answer using only those passages.

**Done when:**
- your chatbot answers **20 test questions** (including **5 that your documents cannot answer**) and gets **at least 80% right**,
- **every answer cites its sources** (document and passage), and unanswerable questions get a clean "I don't know",
- you measured the **token cost** of an answer.

**How to use this notebook**
1. Save your own copy (*File > Save a copy in Drive*). For the free open model, turn on a GPU: *Runtime > Change runtime type > T4 GPU*.
2. Upload the two files your Module 6 notebook saved (`index_chunks.json` and `index_embeddings.npy`). If you do not have them, the notebook rebuilds the index from `my_docs`.
3. Run cells from top to bottom. Fill in every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
4. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page.

## Step 0: Setup and your Module 6 index (given)

In [ ]:
import re, json, time, random
from pathlib import Path
import numpy as np

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

random.seed(0); np.random.seed(0)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")    # the SAME model as Module 6: questions must be embedded like the documents

# ---- the toolbox you built in Module 6 (given, so this notebook stands alone) ----
def embed(texts):
    return embed_model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

def cosine_scores(query_vec, matrix):
    dots = matrix @ query_vec
    norms = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vec)
    return dots / (norms + 1e-12)

def top_k(query_vec, matrix, k=3):
    scores = cosine_scores(query_vec, matrix)
    order = np.argsort(scores)[::-1][:k]
    return order, scores[order]

def chunk_text(text, max_words=60):
    chunks, current, count = [], [], 0
    for para in [p.strip() for p in text.split("\n\n") if p.strip()]:
        words = para.split()
        if len(words) > max_words:
            if current:
                chunks.append("\n\n".join(current)); current, count = [], 0
            for i in range(0, len(words), max_words):
                chunks.append(" ".join(words[i:i + max_words]))
            continue
        if count + len(words) > max_words and current:
            chunks.append("\n\n".join(current)); current, count = [], 0
        current.append(para); count += len(words)
    if current:
        chunks.append("\n\n".join(current))
    return chunks
print("Toolbox ready.")

In [ ]:
USE_DEMO = False       # True: use the demo collection (Python standard-library documentation)
DOCS_DIR = Path("my_docs")
import inspect, importlib
DEMO_MODULES = ["re", "json", "math", "statistics", "random", "itertools", "collections", "datetime", "textwrap",
                "heapq", "bisect", "functools", "shutil", "csv", "urllib.parse", "os.path", "string", "fnmatch",
                "glob", "difflib", "zipfile", "base64", "hashlib", "time", "operator"]

def make_demo_docs(folder="my_docs"):
    """Write one text file per standard-library module, each holding the docstrings of its public functions and classes."""
    Path(folder).mkdir(exist_ok=True)
    total = 0
    for name in DEMO_MODULES:
        mod = importlib.import_module(name)
        label = name
        paras = []
        for attr, obj in sorted(vars(mod).items()):
            if attr.startswith("_") or not (inspect.isfunction(obj) or inspect.isclass(obj) or inspect.isbuiltin(obj)):
                continue
            doc = inspect.getdoc(obj)
            if not doc: continue
            doc = " ".join(doc.split())
            words = doc.split()
            if len(words) < 8: continue
            doc = " ".join(words[:60])
            paras.append(f"{label}.{attr}: {doc}")
        Path(folder, f"{name.replace('.', '_')}.txt").write_text("\n\n".join(paras), encoding="utf-8")
        total += len(paras)
    return total


if Path("index_chunks.json").exists() and Path("index_embeddings.npy").exists():
    data = json.loads(Path("index_chunks.json").read_text(encoding="utf-8"))
    chunk_texts, chunk_sources = data["texts"], data["sources"]
    chunk_embs = np.load("index_embeddings.npy")
    print(f"Loaded your Module 6 index: {len(chunk_texts)} chunks, embeddings {chunk_embs.shape}")
else:
    print("No saved index found: rebuilding it from my_docs.")
    if USE_DEMO:
        import shutil
        shutil.rmtree(DOCS_DIR, ignore_errors=True); make_demo_docs(str(DOCS_DIR))
    assert DOCS_DIR.exists(), "No index files and no my_docs folder. Upload your Module 6 index files, upload my_docs, or set USE_DEMO = True."
    chunk_texts, chunk_sources = [], []
    for path in sorted(DOCS_DIR.rglob("*")):
        if path.suffix.lower() in (".txt", ".md") and path.is_file():
            for c in chunk_text(path.read_text(encoding="utf-8", errors="ignore"), 60):
                chunk_texts.append(c); chunk_sources.append(path.name)
    chunk_embs = embed(chunk_texts)
    print(f"Built an index: {len(chunk_texts)} chunks, embeddings {chunk_embs.shape}")
assert len(chunk_texts) == len(chunk_sources) == len(chunk_embs), "The chunk list and the embedding matrix must have the same length."

## Step 1: Connect a language model (given)

RAG needs a language model to write the answer. Choose one with `LLM_PROVIDER`:

- **`"local"`** (the default): a small **open model** that runs in your Colab session. Free, no account. Needs a GPU runtime. The first call downloads the model (a few minutes).
- **`"anthropic"`**: a hosted **Claude** model through the API. Better quality, but it needs an API key and costs a little money per question. In Colab, click the key icon (**Secrets**), add a secret named `ANTHROPIC_API_KEY`, and switch on notebook access.

Both are called through one function, `generate(system, user)`, so the rest of the notebook does not care which you picked. It returns the answer text and how many **tokens** went in and out, because tokens are what you pay for.

In [ ]:
LLM_PROVIDER = "local"                      # "local" (free, open model) or "anthropic" (hosted, needs an API key)
LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # a small open chat model; a larger one (for example the 3B version) follows instructions better
API_MODEL = "claude-haiku-4-5-20251001"
_llm = {}

def generate(system, user, max_tokens=300):
    """Ask the language model. Returns {"text": ..., "input_tokens": ..., "output_tokens": ...}."""
    if LLM_PROVIDER == "anthropic":
        if "client" not in _llm:
            import anthropic
            from google.colab import userdata
            _llm["client"] = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
        r = _llm["client"].messages.create(model=API_MODEL, max_tokens=max_tokens, system=system,
                                           messages=[{"role": "user", "content": user}])
        return {"text": r.content[0].text.strip(), "input_tokens": r.usage.input_tokens, "output_tokens": r.usage.output_tokens}
    if "pipe" not in _llm:
        from transformers import pipeline
        _llm["pipe"] = pipeline("text-generation", model=LOCAL_MODEL, device_map="auto", torch_dtype="auto")
    pipe = _llm["pipe"]
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    out = pipe(messages, max_new_tokens=max_tokens, do_sample=False)
    text = out[0]["generated_text"][-1]["content"].strip()
    prompt_text = pipe.tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return {"text": text,
            "input_tokens": len(pipe.tokenizer(prompt_text)["input_ids"]),
            "output_tokens": len(pipe.tokenizer(text)["input_ids"])}

print("Language model:", LLM_PROVIDER, "-", API_MODEL if LLM_PROVIDER == "anthropic" else LOCAL_MODEL)

### The problem RAG solves: ask without your documents (given)

First ask the model a question about **your** documents, with **no retrieval at all**. A model knows a lot, but it has never seen your private documents, and it will not always admit that.

In [ ]:
no_rag_question = ("How do I pause the program for a few seconds?" if USE_DEMO
                   else "Write a question about something that only YOUR documents explain.")
reply = generate("You are a helpful assistant.", no_rag_question)
print("QUESTION:", no_rag_question)
print("ANSWER WITHOUT RETRIEVAL:", reply["text"])
print(f"({reply['input_tokens']} tokens in, {reply['output_tokens']} tokens out)")

**Look at the answer.** Is it about *your* documents or about something generic? Could you check where it came from? Write what you notice:

*Your notes:*

## Step 2: Retrieve

Write `retrieve(question, k)`. It embeds the question, finds the `k` most similar chunks with your `top_k`, and returns a list of dictionaries, **best first**, each with these keys:
- `"id"`: the chunk's index in `chunk_texts`,
- `"source"`: the document name (from `chunk_sources`),
- `"text"`: the chunk text,
- `"score"`: the cosine similarity (a plain Python `float`).

*Hint:* `embed([question])[0]` is the question vector, and `top_k(vector, chunk_embs, k)` gives indices and scores.

In [ ]:
def retrieve(question, k=4):
    """Return the k most similar chunks as a list of dicts with keys id, source, text, score (best first)."""
    # YOUR CODE HERE
    return []

### Self-check: retrieve

In [ ]:
_r = retrieve(chunk_texts[5], k=3)
assert len(_r) == 3, f"Expected 3 results, got {len(_r)}."
assert all(set(p) >= {"id", "source", "text", "score"} for p in _r), "Each result needs the keys id, source, text and score."
assert _r[0]["id"] == 5, f"Asking with the exact text of chunk 5 should return chunk 5 first, but got chunk {_r[0]['id']}."
assert _r[0]["score"] > 0.99, f"The exact text of a chunk should have a score near 1 but got {_r[0]['score']:.2f}."
assert _r[0]["source"] == chunk_sources[5] and _r[0]["text"] == chunk_texts[5], "source and text must come from the chunk with that id."
assert all(_r[i]["score"] >= _r[i + 1]["score"] for i in range(2)), "Results must be sorted best first."
assert isinstance(_r[0]["score"], float) and isinstance(_r[0]["id"], int), "Use plain Python float and int (float(...), int(...))."
_j = next((i for i, s in enumerate(chunk_sources) if s != chunk_sources[0]), 5)
_rj = retrieve(chunk_texts[_j], k=2)[0]
assert _rj["id"] == _j and _rj["source"] == chunk_sources[_j], f"The source must be the document name of the retrieved chunk: expected {chunk_sources[_j]!r} but got {_rj['source']!r}."
print("retrieve() check passed.")

## Step 3: The prompt

The prompt is where you **control hallucination**. Write `build_prompt(question, passages)`. It returns `(system, user)`, two strings.

The **system** message sets the rules. It must tell the model to:
1. answer **only** from the numbered passages (no outside knowledge),
2. **cite** the passages it used by number in square brackets, such as `[1]` or `[2]`,
3. reply with **exactly** the sentence stored in `ABSTAIN` if the passages do not contain the answer,
4. be brief.

The **user** message holds the evidence and the question, in this order: each passage as `[1] (source: file name)` on one line followed by its text on the next, passages separated by a blank line, then a blank line and `QUESTION: ...`.

In [ ]:
ABSTAIN = "I don't know based on the documents."

def build_prompt(question, passages):
    """Return (system, user): the rules for the model, and the numbered passages followed by the question."""
    system = ""
    user = ""
    # YOUR CODE HERE
    return system, user

### Self-check: build_prompt

In [ ]:
_ps = [{"source": "a.txt", "text": "Alpha text"}, {"source": "b.txt", "text": "Beta text"}]
_sys, _usr = build_prompt("What is X?", _ps)
assert _sys and _usr, "build_prompt() must return two non-empty strings."
assert "[1]" in _usr and "[2]" in _usr, "Number the passages [1], [2], ... in the user message."
assert "a.txt" in _usr and "b.txt" in _usr and "Alpha text" in _usr and "Beta text" in _usr, "Include every passage's source and text."
assert _usr.index("Alpha text") < _usr.index("Beta text") < _usr.index("What is X?"), "Order: passages first (in order), then the question."
assert "QUESTION:" in _usr, "Label the question with QUESTION:"
assert ABSTAIN in _sys, "The system message must contain the exact sentence stored in ABSTAIN, so the model knows what to say."
assert "only" in _sys.lower(), "Tell the model to answer ONLY from the passages."
assert "cite" in _sys.lower() or "brackets" in _sys.lower(), "Tell the model to cite passages by number in square brackets."
print("build_prompt() check passed.")

## Step 4: Citations and the full answer

**First**, write `extract_citations(text, passages)`. The model will write things like `The module is re. [2]` or even `[1, 3]`. Return the list of passage dictionaries the answer **cites**:
- only numbers that exist (a number `n` points to `passages[n - 1]`; ignore `0` and numbers that are too large, because models sometimes invent them),
- each passage **once**, in the order of first appearance,
- support both `[2]` and lists such as `[1, 3]`.

*Hint:* `re.findall(r"\[([\d,\s]+)\]", text)` finds the contents of brackets that hold only digits, commas and spaces.

In [ ]:
def extract_citations(text, passages):
    """Return the passages cited in the text, as a list without repeats, in order of first appearance."""
    cited = []
    # YOUR CODE HERE
    return cited

### Self-check: extract_citations

In [ ]:
_ps = [{"id": 10}, {"id": 11}, {"id": 12}, {"id": 13}]
_c = extract_citations("Use [2] and [1], also [2] again and [7] and [0].", _ps)
assert [p["id"] for p in _c] == [11, 10], f"Expected the passages with ids [11, 10] (cited in that order, no repeats, ignoring [7] and [0]) but got {[p['id'] for p in _c]}."
_c2 = extract_citations("See [1, 3] for details.", _ps)
assert [p["id"] for p in _c2] == [10, 12], f"A list like [1, 3] should cite passages 1 and 3 (ids [10, 12]) but got {[p['id'] for p in _c2]}."
assert extract_citations("No brackets at all.", _ps) == [], "No citations in the text should give an empty list."
print("extract_citations() check passed.")

**Now the full pipeline.** Write `answer(question, k=4, min_score=0.0)`. It returns a dictionary with the keys `question`, `answer`, `abstained`, `citations`, `passages`, `input_tokens`, `output_tokens`. The steps:

1. `passages = retrieve(question, k)`.
2. **Gate:** if there are no passages, or the **best score is below `min_score`**, do **not** call the model. Return `answer = ABSTAIN`, `abstained = True`, `citations = []`, and zero tokens. (Cheaper, and one less chance to hallucinate.)
3. Otherwise build the prompt, call `generate(system, user)`, and read the text.
4. The answer is an **abstention** if the text, after `strip().lower()`, starts with `"i don't know"`.
5. `citations` are the passages the text cites with `extract_citations`. For an abstention they must be an empty list.

In [ ]:
def answer(question, k=4, min_score=0.0):
    """Retrieve, prompt, generate, and package the result (see the steps above)."""
    passages = retrieve(question, k)
    # YOUR CODE HERE
    return {"question": question, "answer": ABSTAIN, "abstained": True, "citations": [], "passages": passages,
            "input_tokens": 0, "output_tokens": 0}

### Self-check: answer (with a pretend language model, so it is free and instant)

In [ ]:
_real_generate = generate
_calls = []
def _fake_reply(text):
    def fake(system, user, max_tokens=300):
        _calls.append(1)
        return {"text": text, "input_tokens": 120, "output_tokens": 15}
    return fake
try:
    generate = _fake_reply("The first passage explains it. [2] Also see [9].")
    _r = answer(chunk_texts[3], k=3)
    assert _r["answer"] == "The first passage explains it. [2] Also see [9].", "answer must hold the model's text."
    assert _r["abstained"] is False and len(_r["citations"]) == 1 and _r["citations"][0] is _r["passages"][1], "The model cited [2] only (the [9] does not exist), so citations should be exactly the second retrieved passage."
    assert (_r["input_tokens"], _r["output_tokens"]) == (120, 15), "Copy the token counts from the model's reply."
    assert set(_r) >= {"question", "answer", "abstained", "citations", "passages", "input_tokens", "output_tokens"}, "Missing keys in the result dictionary."

    generate = _fake_reply("I don't know based on the documents. [2]")
    _r = answer(chunk_texts[3], k=3)
    assert _r["abstained"] is True and _r["citations"] == [], "When the model says it does not know, abstained must be True and citations must be empty, even if the text happens to contain a bracketed number."

    _calls.clear(); generate = _fake_reply("Should not be called [1].")
    _r = answer(chunk_texts[3], k=3, min_score=1.5)        # no score can reach 1.5
    assert _calls == [] and _r["abstained"] is True and _r["answer"] == ABSTAIN and (_r["input_tokens"], _r["output_tokens"]) == (0, 0), \
        "When the best score is below min_score, answer() must NOT call the model: return the abstention with zero tokens."
    _calls.clear(); answer(chunk_texts[3], k=3, min_score=0.0)
    assert len(_calls) == 1, "With min_score=0 the model should be called exactly once."
finally:
    generate = _real_generate
print("answer() check passed.")

## Step 5: Try it, with sources (given)

`ask()` prints the answer and its sources. **Try 5 questions of your own**, including one your documents cannot answer. (The first call with the local model downloads it and takes a few minutes.)

In [ ]:
def ask(question, k=4, min_score=0.0):
    r = answer(question, k, min_score)
    print("QUESTION:", question)
    print("ANSWER:  ", r["answer"])
    if r["abstained"]:
        print("SOURCES:  (none: the chatbot said it does not know)")
    elif not r["citations"]:
        print("SOURCES:  WARNING - the answer cites nothing")
    for c in r["citations"]:
        print(f"SOURCE:   [{c['source']}] {c['text'][:140].replace(chr(10), ' ')}...")
    print(f"TOKENS:   {r['input_tokens']} in, {r['output_tokens']} out\n")
    return r

_ = ask("How do I pause the program for a few seconds?" if USE_DEMO else "Write a question about your documents here")

**Your experiments:** ask 5 questions and watch for: a correct, cited answer; a wrong answer; an answer with no citation; and a question that should get "I don't know". What did the chatbot get wrong, and was the problem in the **retrieval** (wrong passages) or the **generation** (the model ignored the passages)? Write what you found:

*Your notes:*

## Step 6: A test set

Write **20 test questions**: **15 that your documents can answer** and **5 that they cannot**. Each is a dictionary:

```python
{"question": "...", "gold": ["phrase that marks the right passage"], "answerable": True}
{"question": "What is the capital of France?", "gold": [], "answerable": False}
```

- For answerable questions, `gold` is a phrase that appears in the **right passage** and **not in the question** (as in Module 6).
- For unanswerable questions, `gold` is empty and `answerable` is `False`. Pick questions that are plausible but really not covered. These test your hallucination control: the chatbot must say "I don't know".

With the demo documents a ready-made set called `DEMO_TESTS` is provided.

In [ ]:
DEMO_EVAL = [
    {"query": "cut a string into pieces wherever a pattern occurs", "gold": ["re.split:"]},
    {"query": "average of a list of numbers", "gold": ["statistics.mean:", "statistics.fmean:"]},
    {"query": "find the value that occurs most often in data", "gold": ["statistics.mode:", "statistics.multimode:"]},
    {"query": "running totals of a sequence", "gold": ["itertools.accumulate:"]},
    {"query": "every possible ordering of the elements", "gold": ["itertools.permutations:"]},
    {"query": "count how often each item occurs", "gold": ["collections.Counter:"]},
    {"query": "turn a JSON text into Python data", "gold": ["json.loads:", "json.load:"]},
    {"query": "convert a Python object to text in JSON format", "gold": ["json.dumps:", "json.dump:"]},
    {"query": "read rows from a comma separated file", "gold": ["csv.reader:"]},
    {"query": "break long text into lines of a maximum width", "gold": ["textwrap.wrap:", "textwrap.fill:"]},
    {"query": "the few smallest values from a large dataset", "gold": ["heapq.nsmallest:"]},
    {"query": "add an element to a sorted list and keep it in order", "gold": ["bisect.insort"]},
    {"query": "copy a whole folder with everything inside it", "gold": ["shutil.copytree:"]},
    {"query": "compute a hash digest of some bytes", "gold": ["hashlib.md5:", "hashlib.sha1:", "hashlib.sha256:", "hashlib.sha512:"]},
    {"query": "how far apart are two moments in time", "gold": ["datetime.timedelta:"]},
    {"query": "does this file or folder exist", "gold": ["os.path.exists:"]},
    {"query": "break a web address into its parts", "gold": ["urllib.parse.urlparse:"]},
    {"query": "remember the results of expensive function calls", "gold": ["functools.lru_cache:"]},
    {"query": "find files whose names match a wildcard", "gold": ["glob.glob:", "fnmatch.fnmatch:"]},
    {"query": "pause the program for a few seconds", "gold": ["time.sleep:"]},
]

DEMO_TESTS = ([{"question": q["query"], "gold": q["gold"], "answerable": True} for q in DEMO_EVAL[:15]] +
              [{"question": "What is the capital of France?", "gold": [], "answerable": False},
               {"question": "Who wrote the play Hamlet?", "gold": [], "answerable": False},
               {"question": "How do I train a neural network with PyTorch?", "gold": [], "answerable": False},
               {"question": "What is the population of Canada?", "gold": [], "answerable": False},
               {"question": "How do I install numpy with pip?", "gold": [], "answerable": False}])

test_set = []   # YOUR CODE HERE: 20 dictionaries (15 answerable, 5 not). With the demo documents you may use DEMO_TESTS.
print(len(test_set), "test questions")

### Self-check: test set

In [ ]:
assert len(test_set) >= 20, f"You have {len(test_set)} test questions. Write at least 20."
_unans = [t for t in test_set if not t["answerable"]]
assert len(_unans) >= 5, f"You have {len(_unans)} unanswerable questions. Include at least 5."
_all_text = " ".join(chunk_texts).lower()
for _i, _t in enumerate(test_set):
    assert _t.get("question") and "answerable" in _t, f"Test {_i + 1} needs a 'question' and an 'answerable' flag."
    if _t["answerable"]:
        assert _t["gold"], f"Answerable test {_i + 1} needs at least one gold phrase."
        for _g in _t["gold"]:
            assert _g.lower() not in _t["question"].lower(), f"Gold phrase {_g!r} appears in its own question. Choose a phrase that is not in the question."
            assert _g.lower() in _all_text, f"Gold phrase {_g!r} (test {_i + 1}) is not in any chunk. Check the spelling, or make sure your documents contain the answer."
    else:
        assert not _t["gold"], f"Test {_i + 1} is marked unanswerable, so its gold list must be empty."
print(f"Test set check passed: {len(test_set)} questions, {len(_unans)} unanswerable.")

## Step 7: Grading

Write `grade(item, result)`, which returns `True` or `False` for one test question (`result` is what `answer()` returned):

- **Answerable question:** correct if the chatbot **did not abstain** and **at least one of the passages it cited contains a gold phrase** (ignore upper and lower case). So the answer has to be both **grounded** (it cites something) and cite the **right source**.
- **Unanswerable question:** correct if the chatbot **abstained**.

*This is an automatic proxy: it checks the sources, not the wording of the answer. In Step 9 you also read some answers yourself.*

In [ ]:
def grade(item, result):
    """True if the chatbot handled this test question correctly."""
    # YOUR CODE HERE
    return False

### Self-check: grade

In [ ]:
_cite = [{"text": "The Sleep function delays execution."}]
_ans = {"question": "q", "gold": ["sleep"], "answerable": True}
_no = {"question": "q", "gold": [], "answerable": False}
assert grade(_ans, {"abstained": False, "citations": _cite}) is True, "A cited passage that contains the gold phrase (in any letter case) is correct."
assert grade(_ans, {"abstained": False, "citations": [{"text": "Unrelated text."}]}) is False, "Citing a passage without the gold phrase is wrong."
assert grade(_ans, {"abstained": False, "citations": []}) is False, "An answer that cites nothing cannot be correct."
assert grade(_ans, {"abstained": False, "citations": [{"text": "Unrelated text."}], "passages": _cite}) is False, "Only the passages the answer CITES count, not every passage that was retrieved."
assert grade(_ans, {"abstained": True, "citations": []}) is False, "Abstaining on an answerable question is wrong."
assert grade(_no, {"abstained": True, "citations": []}) is True, "Abstaining on an unanswerable question is correct."
assert grade(_no, {"abstained": False, "citations": _cite}) is False, "Answering an unanswerable question is a hallucination: wrong."
print("grade() check passed.")

## Step 8: Choose the gate (`min_score`)

The gate stops the model from even trying when retrieval found nothing similar. Where should the line be? Look at the **best retrieval score for each test question**: answerable questions should score higher than unanswerable ones. This cell is given and needs no language model.

In [ ]:
import matplotlib.pyplot as plt
best_scores = [(t, retrieve(t["question"], 1)[0]["score"]) for t in test_set]
ans_s = [s for t, s in best_scores if t["answerable"]]; un_s = [s for t, s in best_scores if not t["answerable"]]
plt.figure(figsize=(7, 2.6))
plt.scatter(ans_s, [1] * len(ans_s), label="answerable", s=40); plt.scatter(un_s, [0] * len(un_s), label="unanswerable", s=40, c="tab:orange")
plt.yticks([0, 1], ["unanswerable", "answerable"]); plt.xlabel("best retrieval score"); plt.title("Where would you put the line?"); plt.show()
print(f"answerable:   lowest {min(ans_s):.2f}, average {np.mean(ans_s):.2f}")
print(f"unanswerable: highest {max(un_s):.2f}, average {np.mean(un_s):.2f}")

In [ ]:
MIN_SCORE = None    # YOUR CODE HERE: a number between 0 and 1

### Self-check: MIN_SCORE

In [ ]:
assert MIN_SCORE is not None and 0.0 <= MIN_SCORE <= 1.0, "Set MIN_SCORE to a number between 0 and 1."
_blocked = [t["question"] for t, s in best_scores if t["answerable"] and s < MIN_SCORE]
assert len(_blocked) <= 2, f"MIN_SCORE={MIN_SCORE} would block {len(_blocked)} answerable questions before the model even sees them, for example {_blocked[0]!r}. Lower the line."
_caught = sum(1 for t, s in best_scores if not t["answerable"] and s < MIN_SCORE)
print(f"MIN_SCORE = {MIN_SCORE}: blocks {len(_blocked)} answerable question(s) and catches {_caught} of {len(un_s)} unanswerable ones before calling the model.")

## Step 9: Run the real test

Run all 20 questions. This calls the language model once per question (unless the gate stops it first). This cell is given. **Read the output**: look at each answer, not just the score.

In [ ]:
def run_tests(test_set, k=4, min_score=0.0):
    results, ok = [], 0
    for item in test_set:
        r = answer(item["question"], k, min_score)
        good = grade(item, r); ok += good
        results.append((item, r, good))
        tag = "PASS" if good else "FAIL"
        cites = ",".join(str(p["id"]) for p in r["citations"]) or "-"
        print(f"{tag}  {'answerable  ' if item['answerable'] else 'unanswerable'}  cites[{cites}]  {r['answer'][:90].replace(chr(10), ' ')}")
    return results, ok / len(test_set)

results, accuracy = run_tests(test_set, k=4, min_score=MIN_SCORE)
in_tok = sum(r["input_tokens"] for _, r, _ in results); out_tok = sum(r["output_tokens"] for _, r, _ in results)
print(f"\nAccuracy: {accuracy:.0%}   (target: 80% or more)")
print(f"Tokens for {len(results)} questions: {in_tok:,} in, {out_tok:,} out")

### Self-check: every answer cites a source

In [ ]:
_uncited = [r["question"] for _, r, _ in results if not r["abstained"] and not r["citations"]]
assert not _uncited, (f"{len(_uncited)} answers cite no source, for example {_uncited[0]!r}. Strengthen the instruction in build_prompt() to cite passage numbers like [1], "
                      "or use a larger model, and run again.")
print("Citation check passed: every answer that was not an abstention cites at least one source.")

### Self-check: accuracy

In [ ]:
assert accuracy >= 0.8, (f"Accuracy is {accuracy:.0%}, below the 80% target. Look at the FAIL lines above and decide for each whether the right passage was not retrieved "
                         "(try a larger k or a better MIN_SCORE), the model ignored the passages (make build_prompt() stricter, or use a larger model), or the test question or its gold phrase is faulty.")
_unans_ok = sum(1 for it, r, g in results if not it["answerable"] and g)
print(f"Accuracy check passed: {accuracy:.0%}. Unanswerable questions handled with a clean 'I don't know': {_unans_ok} of {sum(1 for it, _, _ in results if not it['answerable'])}.")

**Read and judge.** Pick 5 answers and read them yourself. For each: is the answer actually right, and does the cited passage really support it? List the failures and decide for each whether the cause was **retrieval** (the right passage was not in the top k), **generation** (the model ignored or misread the passages), or the **test** (a bad gold phrase). Write your findings:

*Your analysis:*

## Step 10: What does an answer cost?

You pay for **tokens**: pieces of text going into the model (the question **and all the passages**) and coming out (the answer). Write `cost_estimate(input_tokens, output_tokens, price_in_per_m, price_out_per_m)`: the cost in dollars, where the prices are per **million** tokens.

*Example:* 1,000 input tokens at 1.0 dollar per million plus 200 output tokens at 5.0 dollars per million costs 0.001 + 0.001 = 0.002 dollars.

In [ ]:
def cost_estimate(input_tokens, output_tokens, price_in_per_m, price_out_per_m):
    """Cost in dollars. Prices are dollars per million tokens."""
    # YOUR CODE HERE
    return 0.0

### Self-check: cost_estimate

In [ ]:
assert abs(cost_estimate(1000, 200, 1.0, 5.0) - 0.002) < 1e-9, f"1,000 input tokens at 1.0 per million plus 200 output tokens at 5.0 per million should cost 0.002, got {cost_estimate(1000, 200, 1.0, 5.0)}."
assert cost_estimate(0, 0, 1.0, 5.0) == 0, "No tokens should cost nothing."
assert abs(cost_estimate(2_000_000, 0, 3.0, 99.0) - 6.0) < 1e-9, "Prices are per MILLION tokens."
print("cost_estimate() check passed.")

Now apply it. Look up your provider's **current** prices (they change, so check the pricing page) and put them in. For the free local model, leave them at 0. This cell is given.

In [ ]:
PRICE_IN_PER_M = 0.0     # dollars per million INPUT tokens (look up your provider's pricing page)
PRICE_OUT_PER_M = 0.0    # dollars per million OUTPUT tokens

per_q = cost_estimate(in_tok, out_tok, PRICE_IN_PER_M, PRICE_OUT_PER_M) / len(results)
print(f"Average tokens per question: {in_tok / len(results):.0f} in, {out_tok / len(results):.0f} out")
print(f"Average cost per question:   ${per_q:.5f}")
print(f"Cost of 10,000 questions:    ${per_q * 10_000:,.2f}")
print(f"Share of tokens that were INPUT: {in_tok / (in_tok + out_tok):.0%}  (the retrieved passages drive the cost)")

**Question:** which would cut your cost most: fewer passages (`k`), shorter chunks, a cheaper model, or the `min_score` gate? Try changing `k` from 4 to 2 and re-run Step 9: what happens to the accuracy and the tokens?

*Your answer:*

## Stretch challenge (optional)

1. **Log the cost of every answer** in a table (question, tokens in and out, cost) and plot the cost against the number of passages `k`.
2. Add **conversation memory**: keep the last two question-and-answer pairs and include them in the prompt so that follow-up questions like "and what about the second one?" work. Does it hurt retrieval? (Hint: retrieve using the *rewritten* standalone question.)
3. Compare the open model with a larger or hosted one on the same test set: accuracy, citation rate, cost.

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-07/README.md`:
1. What your chatbot knows (your documents) and which language model you used
2. Your accuracy on the 20 questions, and the biggest remaining failure
3. Your cost per question and one idea to reduce it

Then download this notebook (*File > Download > .ipynb*), push it to `module-07` in your portfolio repo (do **not** push private documents or your API key!), and submit the link in Moodle.